# Hotel Booking Cancellation & Overbooking Decision
## 03 — Data Cleaning and Preparation

This notebook cleans a working copy of the raw data. The original CSV remains unchanged. Encoding, scaling and modelling are left for later.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

## 2. Load Raw Dataset

In [ ]:
DATA_PATH = r'C:\\Users\\ASUS\\OneDrive\\Desktop\\AML Project\\hotel_bookings.csv'

df_raw = pd.read_csv(DATA_PATH)
print('Raw shape:', df_raw.shape)
df_raw.head()

## 3. Create Working Copy

In [ ]:
df = df_raw.copy()
print('Working copy:', df.shape)

## 4. Remove Clear Leakage
`reservation_status` and `reservation_status_date` reveal information tied to the final outcome.

In [ ]:
leakage_cols = ['reservation_status', 'reservation_status_date']
df = df.drop(columns=leakage_cols)
print('Shape:', df.shape)

## 5. Missing Values Before Cleaning

In [ ]:
missing = pd.DataFrame({'count':df.isnull().sum(), 'percent':(df.isnull().mean()*100).round(2)})
missing[missing['count']>0].sort_values('percent', ascending=False)

## 6. Handle Missing Values
`children` uses median imputation; `country` becomes `Unknown`; missing agent/company identifiers use 0.

In [ ]:
df['children'] = df['children'].fillna(df['children'].median())
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)
print('Remaining missing values:', df.isnull().sum().sum())

## 7. Zero-Guest Records

In [ ]:
zero_guest = (df['adults']==0) & (df['children']==0) & (df['babies']==0)
print('Zero-guest records:', zero_guest.sum())
df.loc[zero_guest].head()

Bookings with no recorded adults, children or babies are removed from the modelling dataset.

In [ ]:
before=len(df)
df=df.loc[~zero_guest].copy()
print('Rows removed:', before-len(df))
print('Shape:', df.shape)

## 8. Inspect ADR

In [ ]:
df['adr'].describe()

In [ ]:
print('Negative ADR records:', (df['adr']<0).sum())
df[df['adr']<0].head()

Negative ADR is invalid for the intended interpretation, so those records are removed. Large positive values are not automatically treated as invalid.

In [ ]:
before=len(df)
df=df[df['adr']>=0].copy()
print('Negative-ADR rows removed:', before-len(df))

## 9. Duplicate Review

In [ ]:
duplicates=df.duplicated().sum()
print('Duplicate-looking rows:', duplicates)
print('Percentage:', round(duplicates/len(df)*100,2),'%')

Duplicate-looking rows are retained because the dataset has no unique booking ID; identical recorded features do not prove that two records are the same transaction.

## 10. Validate Target

In [ ]:
print(df['is_canceled'].value_counts().sort_index())
print((df['is_canceled'].value_counts(normalize=True).sort_index()*100).round(2))

## 11. Final Checks

In [ ]:
print('Final shape:', df.shape)
print('Missing values:', df.isnull().sum().sum())
print('Zero guests:', ((df['adults']==0)&(df['children']==0)&(df['babies']==0)).sum())
print('Negative ADR:', (df['adr']<0).sum())
print('Leakage columns:', [x for x in ['reservation_status','reservation_status_date'] if x in df.columns])

## 12. Raw vs Cleaned

In [ ]:
print('Raw:', df_raw.shape)
print('Cleaned:', df.shape)
print('Rows removed:', len(df_raw)-len(df))
print('Columns removed:', df_raw.shape[1]-df.shape[1])

## 13. Save Cleaned Dataset
The processed CSV is saved separately; the raw CSV is never overwritten.

In [ ]:
OUTPUT_PATH = Path('../data/processed/hotel_bookings_cleaned.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)
print('Shape:', df.shape)

## Summary

Cleaning performed:
- Removed clear leakage columns
- Handled missing values
- Removed zero-guest records
- Removed negative ADR records
- Retained duplicate-looking rows with justification
- Validated the target and final dataset
- Saved a separate processed CSV

### Next Step
Perform Exploratory Data Analysis (EDA) on the cleaned modelling data.